# Experiment: YOLT-204 Registry Disambiguation Gate

Objective:
- Compare the two local YOLT-204 ClinicalTrials.gov snapshots.
- Decide whether YOLT-204 should be treated as a TDT-specific benchmark, a mixed hemoglobinopathy benchmark, or both.
- Preserve the boundary that registry records are not treatment advice and do not prove efficacy.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

ROOT = Path.cwd()
TDT_SNAPSHOT = (
    ROOT
    / "data/registries/clinicaltrials/2026-04-28-nct06678165-yolt204-tdt-detail.json"
)
MIXED_SNAPSHOT = (
    ROOT / "data/registries/clinicaltrials/2026-04-27-nct07190001-yolt204-detail.json"
)

with TDT_SNAPSHOT.open(encoding="utf-8") as handle:
    tdt_snapshot = json.load(handle)

with MIXED_SNAPSHOT.open(encoding="utf-8") as handle:
    mixed_snapshot = json.load(handle)

tdt_record = tdt_snapshot["records"][0]
mixed_record = mixed_snapshot

tdt_record["nct_id"], mixed_record["nct_id"]

## Gate Rules

- A record is `tdt_specific` only when its title or condition is explicitly transfusion-dependent beta-thalassemia.
- A record is `mixed_or_ambiguous` when it names both TDT and SCD or when its eligibility text is SCD-oriented.
- A record is `results_supported` only if posted or published results are available in the reviewed source packet.
- A watchlist record cannot displace CS-101 or VGB-Ex01 without posted clinical results and clearer mechanistic detail.


In [ ]:
tdt_title = tdt_record["title"].lower()
mixed_title = mixed_record["official_title"].lower()
mixed_conditions = " ".join(mixed_record["conditions"]).lower()
mixed_eligibility = mixed_record["eligibility_excerpt"].lower()

tdt_specific = "transfusion-dependent beta-thalassemia" in tdt_title
mixed_names_tdt_and_scd = "sickle" in mixed_title and "thalassemia" in mixed_title
mixed_eligibility_scd_leaning = (
    "hbss" in mixed_eligibility or "severe scd" in mixed_eligibility
)

classification = {
    "tdt_record": tdt_record["nct_id"],
    "tdt_specific": tdt_specific,
    "mixed_record": mixed_record["nct_id"],
    "mixed_names_tdt_and_scd": mixed_names_tdt_and_scd,
    "mixed_eligibility_scd_leaning": mixed_eligibility_scd_leaning,
    "mixed_conditions": mixed_conditions,
}
classification

## Endpoint Overlap

The comparison separates the practical benchmark role from efficacy claims. Endpoint names are used only as registry design signals.


In [ ]:
def normalize_outcome(value: object) -> str:
    """Return a lowercase outcome label from compact registry data."""
    if isinstance(value, dict):
        return str(value.get("measure", "")).lower()
    return str(value).lower()


tdt_outcomes = [normalize_outcome(item) for item in tdt_record["primary_outcomes"]]
mixed_outcomes = [normalize_outcome(item) for item in mixed_record["primary_outcomes"]]
mixed_outcomes += [
    normalize_outcome(item) for item in mixed_record["secondary_outcomes"]
]

endpoint_flags = {
    "tdt_has_transfusion_endpoint": any("transfusion" in item for item in tdt_outcomes),
    "mixed_has_hbf_endpoint": any(
        "hbf" in item or "fetal" in item for item in mixed_outcomes
    ),
    "mixed_has_f_cell_endpoint": any("f cell" in item for item in mixed_outcomes),
    "mixed_has_intended_modification_endpoint": any(
        "intended" in item for item in mixed_outcomes
    ),
    "mixed_has_scd_endpoint": any("vaso-occlusive" in item for item in mixed_outcomes),
}
endpoint_flags

## Decision

YOLT-204 should be tracked as two registry signals, not collapsed into one claim.


In [ ]:
decision = {
    "benchmark_role": "yolt204_registry_watchlist",
    "tdt_specific_record": tdt_record["nct_id"],
    "mixed_hemoglobinopathy_record": mixed_record["nct_id"],
    "changes_claim": (
        "YOLT-204 is no longer only an ambiguous mixed record; a separate "
        "TDT-specific pilot registry record is present."
    ),
    "does_not_displace": (
        "CS-101 remains the strongest reported Asia editing benchmark, and "
        "VGB-Ex01 remains the closest integrated HbF/alpha-chain/hemolysis "
        "registry comparator."
    ),
    "do_not_claim": [
        "published efficacy",
        "cure",
        "patient eligibility",
        "affordability",
        "replacement for clinician review",
    ],
}
decision